<center><h1>xie_yiling_HW3</h1></center>
<br>
<br>

Name: yiling xie
<br>
Github Username: yilingx-333
<br>
USC ID: 3749312095

## 1. Time Series Classification Part 1: Feature Creation/Extraction

### (a) Download Data

Package imports

In [9]:
from pathlib import Path
import re

import numpy as np
import pandas as pd

Get the AReM Data Set

In [10]:
# Relative path from the notebooks folder to the AReM dataset
data_dir = Path("../data/AReM")

activities = [
    "bending1",
    "bending2",
    "cycling",
    "lying",
    "sitting",
    "standing",
    "walking"
]

signal_columns = [
    "avg_rss12",
    "var_rss12",
    "avg_rss13",
    "var_rss13",
    "avg_rss23",
    "var_rss23"
]

print("Data folder:", data_dir)
print("Folder exists:", data_dir.exists())

def get_dataset_number(file_path):
    """Extract the dataset number from a filename such as dataset12.csv."""
    match = re.search(r"dataset(\d+)\.csv$", file_path.name)

    if match is None:
        raise ValueError(f"Unexpected filename: {file_path.name}")

    return int(match.group(1))


def read_arem_file(file_path):
    """Read and clean one AReM dataset file."""
    data = pd.read_csv(
        file_path,
        comment="#",
        sep=r"[,\s]+",
        engine="python",
        header=None,
        names=["time", *signal_columns],
        usecols=range(7),
        on_bad_lines="skip"
    )

    # Convert all columns to numeric values
    data = data.apply(pd.to_numeric, errors="coerce").dropna()

    if data.empty:
        raise ValueError(f"No valid data found in {file_path}")

    return data


records = []

for activity in activities:
    activity_path = data_dir / activity

    files = sorted(
        activity_path.glob("dataset*.csv"),
        key=get_dataset_number
    )

    for file_path in files:
        records.append({
            "activity": activity,
            "dataset": get_dataset_number(file_path),
            "file_path": file_path,
            "data": read_arem_file(file_path)
        })


instances = pd.DataFrame(records)

print("Total number of instances:", len(instances))
display(
    instances.groupby("activity")
    .size()
    .rename("number_of_instances")
    .to_frame()
)

assert len(instances) == 88

Data folder: ..\data\AReM
Folder exists: True
Total number of instances: 88


,number_of_instances
activity,
bending1,7
bending2,6
cycling,15
lying,15
sitting,15
standing,15
walking,15


### (b) Test and Train Data

In [11]:
def is_test_instance(row):
    """Determine whether an instance belongs to the test set."""
    if row["activity"] in ["bending1", "bending2"]:
        return row["dataset"] in [1, 2]

    return row["dataset"] in [1, 2, 3]


instances["split"] = instances.apply(
    lambda row: "test" if is_test_instance(row) else "train",
    axis=1
)

train_instances = instances[
    instances["split"] == "train"
].reset_index(drop=True)

test_instances = instances[
    instances["split"] == "test"
].reset_index(drop=True)

split_summary = pd.crosstab(
    instances["activity"],
    instances["split"]
)

display(split_summary)

print("Number of training instances:", len(train_instances))
print("Number of test instances:", len(test_instances))

assert len(train_instances) == 69
assert len(test_instances) == 19

split,test,train
activity,,
bending1,2,5
bending2,2,4
cycling,3,12
lying,3,12
sitting,3,12
standing,3,12
walking,3,12


Number of training instances: 69
Number of test instances: 19


### (c) Feature Extraction

#### i. Research

Common time-domain features include minimum, maximum, mean, median, standard deviation, variance, quartiles, range, and interquartile range.

#### ii. Extraction

In [12]:
statistics = {
    "min": lambda x: x.min(),
    "max": lambda x: x.max(),
    "mean": lambda x: x.mean(),
    "median": lambda x: x.median(),
    "std": lambda x: x.std(),
    "1st_quart": lambda x: x.quantile(0.25),
    "3rd_quart": lambda x: x.quantile(0.75)
}


def extract_features(data):
    extracted = {}

    for number, column in enumerate(signal_columns, start=1):
        series = data[column]

        for statistic_name, function in statistics.items():
            extracted[f"{statistic_name}{number}"] = function(series)

    return extracted


feature_rows = []

for instance_number, row in enumerate(
    instances.itertuples(index=False),
    start=1
):
    feature_rows.append({
        "Instance": instance_number,
        "activity": row.activity,
        "dataset": row.dataset,
        "split": row.split,
        **extract_features(row.data)
    })


features = pd.DataFrame(feature_rows)

feature_columns = [
    column for column in features.columns
    if column not in ["Instance", "activity", "dataset", "split"]
]

# Table required in Homework 3: Instance + 42 extracted features
homework_features = features[
    ["Instance", *feature_columns]
]

print("Number of instances:", len(homework_features))
print("Number of extracted features:", len(feature_columns))

display(homework_features.head())

Number of instances: 88
Number of extracted features: 42


,Instance,min1,max1,mean1,median1,std1,1st_quart1,3rd_quart1,min2,max2,...,std5,1st_quart5,3rd_quart5,min6,max6,mean6,median6,std6,1st_quart6,3rd_quart6
0,1,37.25,45.00,40.624792,40.50,1.476967,39.25,42.00,0.0,1.30,...,2.188449,33.0000,36.00,0.0,1.92,0.570583,0.43,0.582915,0.0,1.30
1,2,38.00,45.67,42.812812,42.50,1.435550,42.00,43.67,0.0,1.22,...,1.995255,32.0000,34.50,0.0,3.11,0.571083,0.43,0.601010,0.0,1.30
2,3,35.00,47.40,43.954500,44.33,1.558835,43.00,45.00,0.0,1.70,...,1.999604,35.3625,36.50,0.0,1.79,0.493292,0.43,0.513506,0.0,0.94
3,4,33.00,47.75,42.179812,43.50,3.670666,39.15,45.00,0.0,3.00,...,3.849448,30.4575,36.33,0.0,2.18,0.613521,0.50,0.524317,0.0,1.00
4,5,33.00,45.75,41.678063,41.75,2.243490,41.33,42.75,0.0,2.83,...,2.411026,28.4575,31.25,0.0,1.79,0.383292,0.43,0.389164,0.0,0.50


#### iii. Standard Deviation

In [13]:
def bootstrap_std_ci(values, n_bootstrap=10000, confidence=0.90, seed=42):
    values = np.asarray(values)
    rng = np.random.default_rng(seed)

    bootstrap_stds = []

    for _ in range(n_bootstrap):
        sample = rng.choice(values, size=len(values), replace=True)
        bootstrap_stds.append(np.std(sample, ddof=1))

    alpha = 1 - confidence

    lower = np.quantile(bootstrap_stds, alpha / 2)
    upper = np.quantile(bootstrap_stds, 1 - alpha / 2)

    return np.std(values, ddof=1), lower, upper


results = []

for index, column in enumerate(feature_columns):
    std_value, lower, upper = bootstrap_std_ci(
        features[column],
        seed=42 + index
    )

    results.append({
        "Feature": column,
        "Standard Deviation": std_value,
        "90% CI Lower": lower,
        "90% CI Upper": upper
    })


std_results = pd.DataFrame(results)

display(std_results.round(4))

,Feature,Standard Deviation,90% CI Lower,90% CI Upper
0,min1,9.5700,8.2360,10.7739
1,max1,4.3944,3.3453,5.2902
2,mean1,5.3357,4.7106,5.8772
3,median1,5.4401,4.7962,5.9930
4,std1,1.7722,1.5727,1.9501
5,1st_quart1,6.1536,5.5742,6.6429
6,3rd_quart1,5.1389,4.3356,5.8297
7,min2,0.0000,0.0000,0.0000
8,max2,5.0627,4.6225,5.4077
9,mean2,1.5742,1.3964,1.7058


#### iv. Select Features

I chose minimum, mean, and maximum. The mean shows the typical signal level, while the minimum and maximum show its lower and upper boundaries. These three features give a simple summary of each time series.

## 2. ISLR 3.7.4

### (a) Linear Train

The cubic model should have no larger training RSS than the linear model. Since it has two additional terms, it has more freedom to fit the training observations.

### (b) Linear Test

The linear model will probably have the smaller test RSS. When the real relationship is linear, the extra terms in the cubic model are unnecessary and may reduce its performance on new data.

### (c) Not Linear Train

The cubic model should still have no larger training RSS because it is the more flexible model. This is true even when the exact form of the real relationship is unknown.

### (d) Not Linear Testing

It is impossible to determine from the given information. The better model depends on the shape of the real relationship and the balance between bias and variance.

## 3. ISLR 3.7.3 - Extra Practice 

### (a)

The correct answer is **(iii)**. For fixed GPA and IQ, the salary difference between females and males is:

$$
35 - 10 \times GPA
$$

Therefore, males earn more when \(GPA > 3.5\).

### (b)

For a female with \(GPA=4.0\) and \(IQ=110\):

$$
50+20(4)+0.07(110)+35+0.01(4)(110)-10(4)=137.1
$$

The predicted salary is **$137,100**.

### (c)

False. A small coefficient does not prove that the interaction is unimportant. Its significance depends on its standard error and p-value, as well as the scale of the variables.

## 3. ISLR 3.7.5 - Extra Practice 

For linear regression without an intercept, the estimated coefficient is:

$$
\hat{\beta} = \frac{\sum_{j=1}^{n} x_j y_j}{\sum_{j=1}^{n} x_j^2}
$$

The fitted value is:

$$
\hat{y}_i = x_i\hat{\beta}
$$

Substituting the formula for the coefficient gives:

$$
\hat{y}_i
=
\sum_{j=1}^{n}
\left(
\frac{x_i x_j}{\sum_{k=1}^{n} x_k^2}
\right)y_j
$$

Therefore:

$$
a_j = \frac{x_i x_j}{\sum_{k=1}^{n} x_k^2}
$$

Thus, each fitted value is a linear combination of the observed response values.